In [1]:
import numpy as np
import pandas as pd
import cv2
import os
from PIL import Image
from sklearn.model_selection import train_test_split
from keras.utils import normalize
import tensorflow as tf

In [2]:
image_directory = r'C:\Users\saini\OneDrive\ONLY FOR ME\ai\deeplearning\braintumor\brain_tumor_dataset\\'


no_tumor_images= os.listdir(image_directory + 'no/')
yes_tumor_images= os.listdir(image_directory + 'yes/')
#print(no_tumor_images)


In [52]:
dataset=[]
label=[]
input_size=128


In [53]:
for i, image_name in enumerate(no_tumor_images):
    if(image_name.split('.')[1]=='jpg'):
        image=cv2.imread(image_directory+'no/'+image_name,cv2.IMREAD_GRAYSCALE)
        #image=Image.fromarray(image,'RGB')

        image = cv2.resize(image, (128,128))
        dataset.append(np.array(image))

        #image=image.resize((input_size,input_size))
        #dataset.append(np.array(image))
        label.append(0)


for i, image_name in enumerate(yes_tumor_images):
    if(image_name.split('.')[1]=='jpg'):
        image=cv2.imread(image_directory+'yes/'+image_name,cv2.IMREAD_GRAYSCALE)

        image = cv2.resize(image, (128,128))
        dataset.append(np.array(image))


        #image=Image.fromarray(image,'RGB')
        #image=image.resize((input_size,input_size))
        #dataset.append(np.array(image))
        label.append(1)





In [54]:
print(len(label))

3000


In [55]:
dataset=np.array(dataset)

dataset = np.expand_dims(dataset, axis=-1)
label=np.array(label)

In [56]:
x_train,x_test,y_train,y_test=train_test_split(dataset,label,test_size=0.2,random_state=0)

In [57]:
print(x_train.shape)

(2400, 128, 128, 1)


In [58]:
#reshape= (n,image_width,image_height,n_channel)

In [59]:
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

In [60]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import *

In [61]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

datagen = ImageDataGenerator(
    rotation_range=10,
    zoom_range=0.1,
    horizontal_flip=True
)

In [62]:
datagen.fit(x_train)

In [63]:
model=Sequential()


model.add(Conv2D(32,(3,3),input_shape=(input_size,input_size,1)))
#model.add(BatchNormalization())
model.add(Activation('relu'))
model.add(MaxPooling2D(pool_size=(2,2)))

model.add(Conv2D(32,(3,3),kernel_initializer='he_uniform'))
#model.add(BatchNormalization())
model.add(Activation('relu'))
model.add(MaxPooling2D(pool_size=(2,2)))

model.add(Conv2D(32,(3,3),kernel_initializer='he_uniform'))
#model.add(BatchNormalization())
model.add(Activation('relu'))
model.add(MaxPooling2D(pool_size=(2,2)))

model.add(Flatten())
model.add(Dense(128,Activation('relu')))
model.add(Dropout(0.5))
model.add(Dense(1,Activation('sigmoid')))




C:\Users\saini\AppData\Roaming\Python\Python313\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [64]:
#binary cross entropy loss= 1 dense layer and sigmoid
#cross entropy loss = 2 dense layer(no. of classes) and softmax

In [65]:
model.compile(loss='binary_crossentropy',optimizer='adam',metrics=['accuracy'])

In [66]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=6,
    restore_best_weights=True
)

In [67]:
model.fit(
          datagen.flow(x_train,y_train,batch_size=16),
          verbose=1,
          epochs=50,
          validation_data=(x_test,y_test),
          shuffle=False,
          callbacks=[early_stop])

Epoch 1/50
150/150 ━━━━━━━━━━━━━━━━━━━━ 18s 104ms/step - accuracy: 0.7471 - loss: 0.5276 - val_accuracy: 0.8017 - val_loss: 0.4318
Epoch 2/50
150/150 ━━━━━━━━━━━━━━━━━━━━ 16s 108ms/step - accuracy: 0.8117 - loss: 0.4216 - val_accuracy: 0.8400 - val_loss: 0.3486
Epoch 3/50
150/150 ━━━━━━━━━━━━━━━━━━━━ 15s 97ms/step - accuracy: 0.8438 - loss: 0.3570 - val_accuracy: 0.8200 - val_loss: 0.3978
Epoch 4/50
150/150 ━━━━━━━━━━━━━━━━━━━━ 14s 94ms/step - accuracy: 0.8704 - loss: 0.3091 - val_accuracy: 0.8933 - val_loss: 0.2481
Epoch 5/50
150/150 ━━━━━━━━━━━━━━━━━━━━ 14s 93ms/step - accuracy: 0.8896 - loss: 0.2780 - val_accuracy: 0.8933 - val_loss: 0.2302
Epoch 6/50
150/150 ━━━━━━━━━━━━━━━━━━━━ 14s 94ms/step - accuracy: 0.9042 - loss: 0.2387 - val_accuracy: 0.9200 - val_loss: 0.2233
Epoch 7/50
150/150 ━━━━━━━━━━━━━━━━━━━━ 14s 96ms/step - accuracy: 0.9125 - loss: 0.2265 - val_accuracy: 0.9233 - val_loss: 0.2102
Epoch 8/50
150/150 ━━━━━━━━━━━━━━━━━━━━ 15s 100ms/step - accuracy: 0.9237 - loss: 0.1853

In [68]:
#model.summary()

In [69]:
import numpy as np

print(np.unique(y_train, return_counts=True))
print(np.unique(y_test, return_counts=True))

(array([0, 1]), array([1157, 1243]))
(array([0, 1]), array([343, 257]))


In [70]:
from sklearn.metrics import confusion_matrix

y_pred = (model.predict(x_test) > 0.5).astype(int)
print(confusion_matrix(y_test, y_pred))

19/19 ━━━━━━━━━━━━━━━━━━━━ 1s 64ms/step
[[338   5]
 [  4 253]]


In [71]:
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

y_pred = (model.predict(x_test) > 0.5).astype(int)

print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

19/19 ━━━━━━━━━━━━━━━━━━━━ 1s 53ms/step
[[338   5]
 [  4 253]]
              precision    recall  f1-score   support

           0       0.99      0.99      0.99       343
           1       0.98      0.98      0.98       257

    accuracy                           0.98       600
   macro avg       0.98      0.98      0.98       600
weighted avg       0.99      0.98      0.99       600



In [72]:
from sklearn.metrics import roc_auc_score

y_prob = model.predict(x_test)
print("AUC:", roc_auc_score(y_test, y_prob))

19/19 ━━━━━━━━━━━━━━━━━━━━ 1s 49ms/step
AUC: 0.9958366893171944


In [73]:
print(model.input_shape)

(None, 128, 128, 1)


In [74]:
model.save("brain_tumor_cnn.keras")